# Шаг 5. Универсальная модель

Таблица любого бизнеса превращается в таблицу с одинаковыми колонками из каталога полей `FIELDS`.

In [ ]:
# Подготовка: чтобы ноутбук видел папку app и подхватывал изменения в коде без перезапуска
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

In [ ]:
import json

from app.config import SYNTHETIC_DIR
from app.data import EVENT_COLUMNS, FIELDS, available_fields, clean, load, to_events


def read_mapping(file_name: str) -> dict[str, str]:
    # Эталон разметки колонок из .json рядом с тестовым файлом
    return json.loads((SYNTHETIC_DIR / file_name).with_suffix(".json").read_text(encoding="utf-8"))

## 1. До: у каждого бизнеса свои названия колонок

In [ ]:
cleaned = {}
for file_name in ["shop.xlsx", "cafe.csv", "salon.xlsx"]:
    table = list(load(SYNTHETIC_DIR / file_name).values())[0]
    cleaned[file_name], _ = clean(table, read_mapping(file_name))
    print(f"{file_name:<11} {list(cleaned[file_name].columns)}")

## 2. После to_events: колонки одинаковые

In [ ]:
events = {}
for file_name, table in cleaned.items():
    events[file_name] = to_events(table, read_mapping(file_name))
    print(f"{file_name:<11} {list(events[file_name].columns)}")

print("\nСтандартные колонки:", EVENT_COLUMNS)

In [ ]:
# Кафе в универсальном виде
events["cafe.csv"].head()

In [ ]:
# Салон: нет категории и количества - колонки есть, но пустые (NaN).
# Мастер попал в поле executor. Клиенты и телефоны в модель не попали - для анализа продаж не нужны
events["salon.xlsx"].head()

In [ ]:
# Главное: один и тот же код работает для любого бизнеса
for file_name, table in events.items():
    print(f"{file_name:<11} выручка: {table['amount'].sum():>14,.2f}")

## 3. Каталог полей и доступные поля

In [ ]:
# Каталог: все поля, которые программа умеет анализировать
for field, description in FIELDS.items():
    print(f"{field:<9} - {description}")

In [ ]:
# Какие поля реально есть у каждого бизнеса - от этого будет зависеть, какую аналитику показывать
for file_name, table in events.items():
    print(f"{file_name:<11} {available_fields(table)}")

In [ ]:
# Набор полей можно менять на лету - просто меняем словарь ролей.
# Убираем мастера у салона ("ignore" - колонка не нужна):
salon_mapping = read_mapping("salon.xlsx")
salon_mapping["Мастер"] = "ignore"
print("Без мастера:", available_fields(to_events(cleaned["salon.xlsx"], salon_mapping)))

### Попробуй сломать: проверки в to_events

In [ ]:
# Забыли указать дату - понятная ошибка
to_events(cleaned["cafe.csv"], {"Позиция": "object", "Выручка, руб": "amount"})

In [ ]:
# Две колонки назначены деньгами - непонятно, какую брать, поэтому ошибка
to_events(
    cleaned["cafe.csv"],
    {"Дата чека": "date", "Кол-во, шт": "amount", "Выручка, руб": "amount"},
)

## 4. prepare_events: файл → события одной командой

Загрузка, очистка и приведение к модели внутри. Это точка входа для интерфейса.

In [ ]:
from app.data import prepare_events

# Было: load -> clean -> to_events. Стало: одна команда
cafe_events, cafe_report = prepare_events(SYNTHETIC_DIR / "cafe.csv", read_mapping("cafe.csv"))
cafe_report

In [ ]:
# Все три бизнеса - одной и той же командой
for file_name in ["shop.xlsx", "cafe.csv", "salon.xlsx"]:
    file_events, file_report = prepare_events(SYNTHETIC_DIR / file_name, read_mapping(file_name))
    print(f"{file_name:<11} строк: {len(file_events):>5}  поля: {file_report['поля']}")

In [ ]:
# Несуществующий лист - понятная ошибка со списком листов, которые есть
prepare_events(SYNTHETIC_DIR / "shop.xlsx", read_mapping("shop.xlsx"), sheet="Лист2")